# 통합 전류센싱 회로 검증

기존 MOSFET 모델에서 계산한 24개 전류·온도 조건을
U1·NTC 저항망·U2 통합 회로에 적용한다.

첫 시험에서는 NTC 센서온도 Ts가 MOSFET 접합온도 Tj와 같다고 가정한다.
각 조건의 VDS와 Ts를 회로에 입력하고 V_pre와 V_final을 기록한다.

통합 출력으로 고정 이득 방식, Calibration 적용 방식,
NTC 온도보상과 Calibration을 함께 적용한 방식의 전류 추정 오차를 비교한다.

본 시험은 데이터시트 기반 모델과 LTspice 시뮬레이션을 이용하며,
실제 하드웨어 측정 결과는 아니다.

In [2]:
import pandas as pd

df_cases = pd.read_csv("../data/processed/mosfet_test_cases.csv")

df_cases.head()

,I_true (A),Tc (°C),Tj (°C),Rds (mΩ),Ploss (W),VDS (V),I_fixed (A),Error (A),Error (%)
0,25,25,25.391946,1.493136,0.933210,0.037328,25.052610,0.052610,0.210441
1,25,50,50.444652,1.693913,1.058696,0.042348,28.421358,3.421358,13.685432
2,25,75,75.502747,1.915229,1.197018,0.047881,32.134708,7.134708,28.538830
3,25,100,100.571244,2.176169,1.360106,0.054404,36.512910,11.512910,46.051640
4,25,125,125.642524,2.447710,1.529819,0.061193,41.068965,16.068965,64.275858


In [3]:
df_integrated_inputs = df_cases[
    ["I_true (A)", "Tc (°C)", "Tj (°C)", "VDS (V)"]
].copy()

df_integrated_inputs["Ts (°C)"] = df_integrated_inputs["Tj (°C)"]

df_integrated_inputs = df_integrated_inputs[
    ["I_true (A)", "Tc (°C)", "Tj (°C)", "Ts (°C)", "VDS (V)"]
]

df_integrated_inputs

,I_true (A),Tc (°C),Tj (°C),Ts (°C),VDS (V)
0,25,25,25.391946,25.391946,0.037328
1,25,50,50.444652,50.444652,0.042348
2,25,75,75.502747,75.502747,0.047881
3,25,100,100.571244,100.571244,0.054404
4,25,125,125.642524,125.642524,0.061193
5,25,150,150.721827,150.721827,0.068745
6,50,25,26.577752,26.577752,0.075131
7,50,50,51.791048,51.791048,0.085288
8,50,75,77.027639,77.027639,0.096554
9,50,100,102.304631,102.304631,0.109745


In [4]:
df_integrated_inputs.to_csv(
    "../data/processed/integrated_test_inputs.csv",
    index=False,
    encoding="utf-8-sig"
)

In [5]:
df_inputs_ts_tc = df_integrated_inputs.copy()

df_inputs_ts_tc["Ts (°C)"] = df_inputs_ts_tc["Tc (°C)"]

df_inputs_ts_tc

,I_true (A),Tc (°C),Tj (°C),Ts (°C),VDS (V)
0,25,25,25.391946,25,0.037328
1,25,50,50.444652,50,0.042348
2,25,75,75.502747,75,0.047881
3,25,100,100.571244,100,0.054404
4,25,125,125.642524,125,0.061193
5,25,150,150.721827,150,0.068745
6,50,25,26.577752,25,0.075131
7,50,50,51.791048,50,0.085288
8,50,75,77.027639,75,0.096554
9,50,100,102.304631,100,0.109745


In [6]:
df_inputs_ts_tc.to_csv(
    "../data/processed/integrated_test_inputs_ts_tc.csv",
    index=False,
    encoding="utf-8-sig"
)

## 통합 회로 출력 확인

In [7]:
df_integrated_results = pd.read_csv(
    "../data/processed/integrated_24case_outputs.csv"
)

df_integrated_results.head()

,I_true (A),Tc (°C),Tj (°C),Ts (°C),VDS (V),V_pre (V),V_final (V)
0,25,25,25.391946,25.391946,0.037328,0.074664,0.069829
1,25,50,50.444652,50.444652,0.042348,0.084703,0.071970
2,25,75,75.502747,75.502747,0.047881,0.095769,0.070433
3,25,100,100.571244,100.571244,0.054404,0.108816,0.068387
4,25,125,125.642524,125.642524,0.061193,0.122393,0.067444


In [9]:
df_integrated_check = df_integrated_results.copy()

df_integrated_check["Expected V_pre (V)"] = (
    2 * df_integrated_check["VDS (V)"]
)

df_integrated_check["U1 gain error (%)"] = (
    (
        df_integrated_check["V_pre (V)"]
        - df_integrated_check["Expected V_pre (V)"]
    )
    / df_integrated_check["Expected V_pre (V)"]
    * 100
)

df_integrated_check[
    [
        "I_true (A)",
        "Tc (°C)",
        "VDS (V)",
        "Expected V_pre (V)",
        "V_pre (V)",
        "U1 gain error (%)"
    ]
]

,I_true (A),Tc (°C),VDS (V),Expected V_pre (V),V_pre (V),U1 gain error (%)
0,25,25,0.037328,0.074657,0.074664,0.009941
1,25,50,0.042348,0.084696,0.084703,0.008682
2,25,75,0.047881,0.095761,0.095769,0.007698
3,25,100,0.054404,0.108808,0.108816,0.006919
4,25,125,0.061193,0.122386,0.122393,0.006116
5,25,150,0.068745,0.137491,0.137498,0.005206
6,50,25,0.075131,0.150262,0.150269,0.004524
7,50,50,0.085288,0.170576,0.170583,0.004032
8,50,75,0.096554,0.193109,0.193116,0.003757
9,50,100,0.109745,0.219489,0.219496,0.003188


In [10]:
df_integrated_check["U1 gain error (%)"].abs().max()

0.00994076511824056

Ts=Tj 조건의 통합 회로 출력 24개를 확인하였다. 조건 중복과 빈 값은 없었으며, U1 출력의 예상값 2×VDS 대비 최대 상대오차는 약 0.00994%였다. 현재 시뮬레이션 조건에서 NTC 저항망과 U2 연결 후에도 U1의 2배 증폭 동작이 유지됨을 확인하였다.